# F# interop: where the domain meets C#

## The clean core has a messy edge
The last chapter showed `Decision.fs` as a seven-line discriminated
union and `DecisionEngine.fs` as a single pattern match on a tuple of
three sum types. That's the gorgeous part. That's the part that closes
six of the seven footguns by default behaviour of the language.

But a .NET solution rarely has just one project. There is a C# ASP.NET
controller that needs to deserialize JSON. There is a CSV importer
that needs to take raw strings off a row. There is an audit log writer
that needs flat records to push to BigQuery. There is a C# integration
test calling `new FuelUploadFacade().Classify(...)` because the team
hasn't fully migrated.

`Interop.fs` is where that translation lives. It is one file — 800
lines as of V3 — and it is **mostly `[<CLIMutable>]` records, string
fields, and adapter code**. Every guarantee the domain core earns,
this file re-earns at the boundary.

If the domain is F# being F#, the boundary is F# wearing a C#
costume.

## What the F# project looks like

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 1607.21875 501" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381579481_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381579481_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381579481_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381579481_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381579481_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381579481_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"><g class="cluster" id="Core" data-look="classic"><rect style="fill:#e6f5ff !important;stroke:#0066cc !important;stroke-width:2px !important" x="1280.21875" y="238" width="319" height="196"></rect><g class="cluster-label" transform="translate(1339.71875, 238)"><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Pure F# domain (chapter 05)
</p>
</span>
</div>
</foreignobject></g></g><g class="cluster" id="Boundary" data-look="classic"><rect style="fill:#fff5e6 !important;stroke:#cc8800 !important;stroke-width:2px !important" x="309.765625" y="8" width="920.453125" height="485"></rect><g class="cluster-label" transform="translate(669.9921875, 8)"><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Interop.fs — the costume layer
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M532.065,206L543.145,195.333C554.226,184.667,576.386,163.333,590.967,152.667C605.547,142,612.547,142,616.047,142L619.547,142" id="L_DTO_Parse_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DTO_Parse_0" data-points="W3sieCI6NTMyLjA2NDg2MDYxMTUxMDgsInkiOjIwNn0seyJ4Ijo1OTguNTQ2ODc1LCJ5IjoxNDJ9LHsieCI6NjIzLjU0Njg3NSwieSI6MTQyfV0=" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path><path d="M883.547,142L887.714,142C891.88,142,900.214,142,923.714,165.32C947.214,188.64,985.881,235.28,1005.215,258.601L1024.548,281.921" id="L_Parse_Facade_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Parse_Facade_0" data-points="W3sieCI6ODgzLjU0Njg3NSwieSI6MTQyfSx7IngiOjkwOC41NDY4NzUsInkiOjE0Mn0seyJ4IjoxMDI3LjEwMTIwMDA2NDQzMywieSI6Mjg1fV0=" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path><path d="M857.359,342L865.891,342C874.422,342,891.484,342,903.516,341.869C915.548,341.739,922.549,341.478,926.049,341.347L929.55,341.216" id="L_Ports_Facade_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Ports_Facade_0" data-points="W3sieCI6ODU3LjM1OTM3NSwieSI6MzQyfSx7IngiOjkwOC41NDY4NzUsInkiOjM0Mn0seyJ4Ijo5MzMuNTQ2ODc1LCJ5IjozNDEuMDY3MzcyNjEzNzg1NH1d" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path><path d="M259.766,281L263.932,281C268.099,281,276.432,281,284.766,281C293.099,281,301.432,281,309.099,281C316.766,281,323.766,281,327.266,281L330.766,281" id="L_Caller_DTO_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Caller_DTO_0" data-points="W3sieCI6MjU5Ljc2NTYyNSwieSI6MjgxfSx7IngiOjI4NC43NjU2MjUsInkiOjI4MX0seyJ4IjozMDkuNzY1NjI1LCJ5IjoyODF9LHsieCI6MzM0Ljc2NTYyNSwieSI6MjgxfV0=" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path><path d="M1205.219,287.015L1209.385,285.513C1213.552,284.01,1221.885,281.005,1230.219,279.503C1238.552,278,1246.885,278,1255.219,278C1263.552,278,1271.885,278,1279.592,279.287C1287.299,280.575,1294.379,283.149,1297.919,284.437L1301.46,285.724" id="L_Facade_DE_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Facade_DE_0" data-points="W3sieCI6MTIwNS4yMTg3NSwieSI6Mjg3LjAxNTM5ODA2Njc0MTE3fSx7IngiOjEyMzAuMjE4NzUsInkiOjI3OH0seyJ4IjoxMjU1LjIxODc1LCJ5IjoyNzh9LHsieCI6MTI4MC4yMTg3NSwieSI6Mjc4fSx7IngiOjEzMDUuMjE4NzUsInkiOjI4Ny4wOTA5MDkwOTA5MDkxfV0=" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path><path d="M1305.219,336L1301.052,336C1296.885,336,1288.552,336,1280.219,336C1271.885,336,1263.552,336,1255.219,336C1246.885,336,1238.552,336,1230.885,336C1223.219,336,1216.219,336,1212.719,336L1209.219,336" id="L_DE_Facade_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DE_Facade_0" data-points="W3sieCI6MTMwNS4yMTg3NSwieSI6MzM2fSx7IngiOjEyODAuMjE4NzUsInkiOjMzNn0seyJ4IjoxMjU1LjIxODc1LCJ5IjozMzZ9LHsieCI6MTIzMC4yMTg3NSwieSI6MzM2fSx7IngiOjEyMDUuMjE4NzUsInkiOjMzNn1d" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path><path d="M980.224,387L968.278,393.833C956.331,400.667,932.439,414.333,894.66,421.167C856.88,428,805.214,428,753.547,428C701.88,428,650.214,428,613.06,416.476C575.907,404.951,553.268,381.902,541.948,370.378L530.628,358.854" id="L_Facade_DTO_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Facade_DTO_0" data-points="W3sieCI6OTgwLjIyMzc2MDE5MDIxNzQsInkiOjM4N30seyJ4Ijo5MDguNTQ2ODc1LCJ5Ijo0Mjh9LHsieCI6NzUzLjU0Njg3NSwieSI6NDI4fSx7IngiOjU5OC41NDY4NzUsInkiOjQyOH0seyJ4Ijo1MjcuODI0OTM2MjI0NDg5OCwieSI6MzU2fV0=" marker-end="url(#mermaid-1779381579481_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_DTO_Parse_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Parse_Facade_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Ports_Facade_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Caller_DTO_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Facade_DE_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DE_Facade_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Facade_DTO_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-Caller-0" transform="translate(133.8828125, 281)"><rect class="basic label-container" style="" x="-125.8828125" y="-75" width="251.765625" height="150"></rect><g class="label" style="" transform="translate(-95.8828125, -60)"><rect></rect><foreignobject width="191.765625" height="120">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
C# callers<br>ASP.NET controllers<br>CSV importer<br>integration tests<br>(string DTOs, possibly null)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DTO-1" transform="translate(454.15625, 281)"><rect class="basic label-container" style="" x="-119.390625" y="-75" width="238.78125" height="150"></rect><g class="label" style="" transform="translate(-89.390625, -60)"><rect></rect><foreignobject width="178.78125" height="120">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
[&lt;CLIMutable&gt;] DTOs<br>FuelUploadRequestDto<br>FuelUploadResponseDto<br>ImportBatchRequest<br>(strings, arrays, no DUs)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Parse-2" transform="translate(753.546875, 142)"><rect class="basic label-container" style="" x="-130" y="-99" width="260" height="198"></rect><g class="label" style="" transform="translate(-100, -84)"><rect></rect><foreignobject width="200" height="168">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
parse* functions<br>parseUploadMode<br>parsePreviousAttempt<br>parseVehicleLookup<br>parseDuplicateCheck<br>(string -&gt; Result&lt;DU, error list&gt;)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Ports-3" transform="translate(753.546875, 342)"><rect class="basic label-container" style="" x="-103.8125" y="-51" width="207.625" height="102"></rect><g class="label" style="" transform="translate(-73.8125, -36)"><rect></rect><foreignobject width="147.625" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
IVehicleRepository<br>IDuplicateRepository<br>(repository ports)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Facade-4" transform="translate(1069.3828125, 336)"><rect class="basic label-container" style="" x="-135.8359375" y="-51" width="271.671875" height="102"></rect><g class="label" style="" transform="translate(-105.8359375, -36)"><rect></rect><foreignobject width="211.671875" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
FuelUploadFacade class<br>RepositoryFuelUploadFacade<br>(C#-shaped entry points)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DE-11" transform="translate(1439.71875, 336)"><rect class="basic label-container" style="" x="-134.5" y="-63" width="269" height="126"></rect><g class="label" style="" transform="translate(-104.5, -48)"><rect></rect><foreignobject width="209" height="96">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
DecisionEngine.classifyBatch<br>typed DUs end-to-end<br>no nulls<br>exhaustive matching
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

The blue zone is the part the last chapter celebrated. The orange zone
is what we are about to look at.

Asymmetry: **the domain is five files totaling around 250 lines. The
boundary is one file at 800 lines.** This is a common shape in F#
applications that have to live alongside C#. The domain stays compact;
the boundary absorbs all the costume-wearing.

## `[<CLIMutable>]` — a deliberate hole in the safety guarantee

Here is a real DTO record from `Interop.fs`:

In [ ]:
// fsharp-fuel-engine/FuelUpload.Domain/Interop.fs
[<CLIMutable>]
type FuelUploadRequestDto =
    { UploadMode: string                    // string, not UploadMode DU
      RequireExternalReference: bool
      MinFuelVolumeGallons: decimal
      MaxFuelVolumeGallons: decimal
      MinTotalCost: decimal
      MaxTotalCost: decimal
      AllowFutureTransactions: bool
      ProcessingDate: string                // string, not DateTimeOffset
      HighFuelVolumeWarningGallons: decimal
      HighCostPerGallonWarning: decimal
      StaleTransactionWarningDays: int
      SuspiciousFuelVolumeGallons: decimal
      SuspiciousTotalCost: decimal
      Rows: FuelUploadRowDto array }

Compare to the typed domain version in `Validation.fs`: `UploadMode` is
a four-case DU. `ProcessingDate` is a `DateTimeOffset`. Here both are
plain `string`. The boundary has been deliberately re-typed to the
shape JSON gives you.

What does `[<CLIMutable>]` actually do? Without the attribute, F#
records are:

- **immutable** — once constructed, fields cannot be reassigned;
- **closed to default construction** — you cannot write `new
  FuelUploadRequestDto()` because the record requires every field to
  be supplied at construction;
- **value-equal** — two records with equal fields compare equal.

`[<CLIMutable>]` keeps the equality behaviour but opens the other two:
the compiler also emits a parameterless constructor and settable
property setters so that C# callers, `System.Text.Json`, and
ASP.NET's model binder can do `new FuelUploadRequestDto { UploadMode =
"Retry", ... }` and have it work. The F# code in `Interop.fs` itself
never *uses* those setters — it always constructs records with the
full literal `{ ... }` syntax. But the type, at the CLR level, exposes
a mutable shape.

That is a **deliberately carved hole** in F#'s safety guarantee. A C#
consumer of the DTO can:

- Default-construct it: `new FuelUploadRequestDto()` — every string is
  `null`, every decimal is `0m`. F# would normally forbid this.
- Mutate it after construction: `dto.UploadMode = null`. F# would
  normally forbid this too.
- Pass `null` for any string field through the property setter.

The boundary is forced to assume any string can be null and any field
can be defaulted. Look at the very first helper in the file:

In [ ]:
let private normalize (value: string) =
    if isNull value then
        ""
    else
        value.Replace("_", "").Trim().ToLowerInvariant()

`isNull value` is the explicit null guard. F#'s own strings can never
be null. But these strings came from C# through a `[<CLIMutable>]`
record, so they can.

The interop layer pays for `[<CLIMutable>]` with a guard at every
single string field it reads.

## Re-earning the safety: boundary parsers

Inside the domain, `UploadMode` is a four-case DU and you `match` on
it. At the boundary, `UploadMode` came in as `dto.UploadMode : string`.
Bridging the two takes an explicit parser:

In [ ]:
let private parseUploadMode field value : Result<UploadMode, FuelUploadMappingError list> =
    match normalize value with
    | "normal"               -> Ok UploadMode.Normal
    | "retry"                -> Ok UploadMode.Retry
    | "conservativerecovery" -> Ok UploadMode.ConservativeRecovery
    | "aggressiverecovery"   -> Ok UploadMode.AggressiveRecovery
    | _ ->
        Error
            [ { Code = FuelUploadMappingErrorCode.InvalidUploadMode
                Field = field
                Detail = $"Unsupported upload mode '{value}'." } ]

This is the heart of what the boundary does. **String in, typed
`Result<DU, error list>` out.** Once a string has passed through this
function, the rest of the F# code can treat the value as a proper
`UploadMode` DU and never has to think about case-sensitivity, typos,
or normalisation again. The boundary has *re-earned* the type-system
guarantee that the JSON layer washed away.

There are six of these in the file: `parseUploadMode`,
`parsePreviousAttempt`, `parseVehicleLookup`, `parseDuplicateCheck`,
`parseDate`, plus a parallel set for the CSV importer (`parseImport*`)
that returns the more permissive `FuelImportError` type. Each one
returns `Result<T, error list>` instead of throwing — the boundary
**accumulates** parse errors instead of failing on the first one. A
malformed batch comes back with every field error at once, not just
the first.

The actual mapping function that ties three independent `Result`s
together for one row:

In [ ]:
let private mapRow index (row: FuelUploadRowDto) : Result<FuelRowContext, FuelUploadMappingError list> =
    let prefix = $"rows[{index}]"
    let occurredAt    = parseDate $"{prefix}.occurredAt" row.OccurredAt
    let vehicleLookup = parseVehicleLookup prefix row
    let duplicateCheck = parseDuplicateCheck prefix row

    match occurredAt, vehicleLookup, duplicateCheck with
    | Ok occurredAt, Ok vehicleLookup, Ok duplicateCheck ->
        Ok { Row = { /* fields */ }
             VehicleLookup = vehicleLookup
             DuplicateCheck = duplicateCheck }
    | _ ->
        Error (errorsOf occurredAt @ errorsOf vehicleLookup @ errorsOf duplicateCheck)

`@` is list concatenation. If all three results are `Ok`, build a
`FuelRowContext`. If any are `Error`, concatenate every error list
into one big error list. Three independent parses, all attempted, all
reported. No first-fail-and-bail.

There is no `Result<T, E>` library to import for this. `Result` and
its operators (`Result.map`, `Result.bind`) ship in `FSharp.Core`.

## Adding a new field at the boundary

Suppose product adds a field to the upload request: `BatchReference`
— an optional caller-supplied string to correlate batches in their
logs. How does the change propagate?

**In the domain:** add `BatchReference: string option` (or a
`BatchReference` typed value) to `ValidationConfig` or a new wrapper
record. One file touched. If any pattern match destructures the record
positionally, the compiler flags it.

**At the boundary:** more work.

1. Add `BatchReference: string` to `FuelUploadRequestDto` in
   `Interop.fs`.
2. Decide null-policy at the boundary. Is it optional (allow `null`
   from C#, parse to `None`)? Required (reject `null`)?
3. Add a `parseBatchReference` function returning
   `Result<_, FuelUploadMappingError list>`.
4. Thread the new parse through `toDomainRequest` — extend the
   thirteen-way tuple match.
5. Add the same field to `ImportBatchRequest` and `mapImportedRow`
   for the CSV path.
6. Add the same field to the response DTO if the response should echo
   it back.
7. Re-run any C# consumer projects to regenerate JSON contracts;
   recompile.

The C# consumers of the F# library see DTO changes the moment they
recompile against the new `FuelUpload.Domain.dll`. Source-level: yes.
Behaviour-level: their old code that ignored `BatchReference` still
compiles, because the field is just *new on the record*. There is no
exhaustiveness check at the boundary the way there is inside the
domain.

This is the asymmetry. **Inside the domain, the type system is your
backstop.** Outside, the type system is something you re-impose, by
hand, in `Interop.fs`. The further you walk out the boundary, the
less help you get.

## Where this still leaks (V3 coda)

In V2, `Interop.fs` was already the largest file in the project. V3
made it considerably larger. The current file holds, in one module:

- The original JSON request/response DTOs (`FuelUploadRequestDto`,
  `FuelUploadResponseDto`).
- A second, stricter set of CSV import DTOs (`ImportedFuelRow`,
  `ImportBatchRequest`, `FuelImportError`) that mirrors the first set
  but with all-string fields and parsing happening one layer earlier.
- Repository port interfaces (`IVehicleRepository`,
  `IDuplicateRepository`) with their own error type families
  (`VehicleRepositoryError`, `DuplicateRepositoryError`).
- A second classify path (`toRepositoryDomainRequest`,
  `classifyWithRepositories`) that uses the ports instead of inline
  DTO fields.
- Response serialisation (`toDecisionDto`, `toResponseDto`,
  `warningText`/`rejectionText`/`quarantineText` formatters built on
  `sprintf "%A"`).
- Two facade classes (`FuelUploadFacade`, `RepositoryFuelUploadFacade`)
  exposing all of the above to C# callers.

The V3 scoring report names this directly:

> *"The interop module is large and mixes several adapter concerns;
> `[<CLIMutable>]` records and string DTO fields weaken the otherwise
> strong model for C# callers."* — `docs/v3-results.md`

What is "several adapter concerns" hiding? Three jobs in one file:

1. **JSON wire format** — `FuelUploadRequestDto` / `FuelUploadResponseDto`.
2. **CSV import format** — `ImportBatchRequest` / `ImportedFuelRow`.
3. **Repository ports** — `IVehicleRepository` / `IDuplicateRepository`
   plus their error families.

In a larger codebase these would be three separate modules with
narrower public surfaces. Bundled into one file, they share helper
functions (`normalize`, `require`, `errorsOf`) — which is convenient
— but they also share blast radius: a change to the JSON DTO shape
touches the same file as a change to the CSV import shape.

The `%A` formatters are another leak. Quoting `toDecisionDto`:

In [ ]:
| RowDecision.SkippedDuplicate skipped ->
    decisionDto
        skipped.Row.RowNumber
        "skipped_duplicate"
        ""
        ""
        [||]
        [||]
        [||]
        ($"%A{skipped.Reason}")          // <- prototype-grade output
        ""

`%A{skipped.Reason}` formats the `DuplicateSkipReason` DU using F#'s
generic debug printer. The output is something like
`"RetryModeDuplicateNotRetryable Finalized"` — which is fine for a
prototype, but it ties the response wire format to the internal DU's
*name and shape*. If you rename a DU case for clarity, you've broken
your JSON contract. The fix is a hand-written `match` per DU that
emits stable lowercase strings, the same way `parseUploadMode` does
the reverse mapping. The boundary should be **symmetric** —
hand-mapped both ways — but the V3 implementation took the shortcut
on the response side.

Then there are the empty strings: when a `RowDecision.Accepted` row
becomes a `FuelUploadDecisionDto`, fields like
`DuplicateSkipReason` and `FatalError` are populated with `""`
instead of `null`. That's because `[<CLIMutable>]` strings default to
empty rather than null in the F# `{ ... }` literal — which is *safer*
than `null`, but still requires the consumer to know that `""` means
"absent" rather than "empty string." A more careful boundary would use
`string option` and an opt-in JSON converter, but that's more code
than the prototype needed.

## How this connects to the other engines

Every engine in this book hits the same wall at the boundary. The
shape of the wall differs:

- **Idiomatic C#** has no boundary problem because its domain is
  *already* in C# — but its domain pays for that with the
  thirty-line-per-DU encoding from chapter 04.
- **F# (this chapter)** keeps the domain pristine and pays in one
  large `Interop.fs` file.
- **Haskell** keeps the domain even more pristine and pays at the FFI
  level — you'll see in chapter 08 that the boundary is JSON via
  `aeson`, but everything outside the Haskell runtime is opaque to
  the Haskell type system.
- **Rust** has the boundary as `serde` derive macros and an explicit
  Vec-to-NonEmpty conversion, which we cover in chapter 09.

Part Ib gathers all four engine boundaries onto one page for direct
comparison. If you want to see how the same `BatchReference`-style
extension lands in each of the four, that's the chapter to skip to:

[Compare all four boundaries side-by-side →](../part1b/11-boundary-returns.ipynb)

Otherwise, the next rung up the ladder is the type-driven extreme. F#
made the safe choice the default. Haskell removes the unsafe choice
entirely — no `[<CLIMutable>]` escape hatch, no nullable strings at
the boundary, side effects tracked in the type signature, and
exhaustive matching as a hard error rather than a warning.

[Next chapter →](07-haskell-primer.ipynb)